# MindTrack · Comparación de modelos (Entregable 3)
## Regresión Logística vs Random Forest vs XGBoost

**Autores:** Geraldin Perez · Daniel Hernandez

| Paso | Contenido |
|------|-----------|
| 1 | Carga desde Google Drive y mismas particiones 70/15/15 del Entregable 2 |
| 2 | Entrenamiento de 3 clasificadores con manejo de desbalance |
| 3 | Ajuste de umbral en **Validation** (Recall clase 1 ≥ 0,85) |
| 4 | Gráficas comparativas y tabla de métricas |
| 5 | Selección del ganador + evaluación final en **Test** |
| 6 | Guardado de resultados y modelo en **Google Drive** |

**Requisito previo:** ejecutar con el mismo `academic_risk_dataset.csv` y `random_state=42` que el notebook de EDA.


---
### Mapa de notebooks · Proyecto MindTrack

| Entregable | Archivo | Qué contiene |
|:---:|:---|:---|
| **2** | `MindTrack_EDA_Preprocesamiento.ipynb` | EDA, validación, preprocesamiento, partición y baseline LR |
| **3** | `MindTrack_03_Comparacion_Modelos.ipynb` | **Este notebook:** comparación LR / RF / XGBoost |

**Usted está en el Entregable 3.**  
No repite el EDA completo ni las gráficas de la presentación del corte 2; asume el mismo pipeline definido en el [Notebook 2](https://colab.research.google.com/gist/Danielhsk/8c4e3e85e3efde9c60b9db408a507afb/MindTrack_EDA_Preprocesamiento.ipynb).

Ejecute primero el Entregable 2 para el informe de datos; use este notebook para el análisis comparativo de modelos.


[![Abrir Entregable 3 en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/Danielhsk/92fab1e431edaeb0ce998bb03a09d04c/MindTrack_03_Comparacion_Modelos.ipynb)

> **Solo comparación de modelos** (LR, RF, XGBoost). El EDA y las gráficas del segundo corte están en el [Entregable 2](https://colab.research.google.com/gist/Danielhsk/8c4e3e85e3efde9c60b9db408a507afb/MindTrack_EDA_Preprocesamiento.ipynb).

Guarde copias en Drive: `Sistemas inteligentes / Notebooks/`. Resultados del modelo: `.../Artefactos/`.


In [ ]:
import warnings
from datetime import datetime
from pathlib import Path

from IPython.display import display

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)


In [ ]:
RANDOM_STATE = 42
RECALL_TARGET = 0.85

DATA_FILENAME = "academic_risk_dataset.csv"
# Ruta en Google Drive (Mi unidad)
DRIVE_DATA_PATH = (
    "/content/drive/MyDrive/Sistemas inteligentes/DataSet/academic_risk_dataset.csv"
)
DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Sistemas inteligentes/Artefactos"
)

TARGET = "Academic_Failure_Risk"
ID_COLS = ["Student_ID"]
LEAKAGE_COLS = ["Academic_Performance_Score"]
NOMINAL_COLS = ["Gender", "Education_Level"]
ORDINAL_COL = "Burnout_Level"
BURNOUT_ORDER = ["Low", "Moderate", "High", "Severe"]
BURNOUT_MAP = {level: code for code, level in enumerate(BURNOUT_ORDER)}
DROP_FIRST_DUMMY = True
EXPECTED_N_FEATURES = 13

MODEL_LABELS = {
    "logistic_regression": "Regresión Logística",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}
MODEL_COLORS = {
    "Regresión Logística": "#1F3A5F",
    "Random Forest": "#475569",
    "XGBoost": "#B45309",
}

MT_NAVY = "#1F3A5F"
MT_SLATE = "#475569"
MT_ACCENT = "#B45309"
MT_MIST = "#94A3B8"
MT_INK = "#0F172A"


In [ ]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess

    subprocess.check_call(
        ["pip", "install", "-q", "scikit-learn", "xgboost", "seaborn", "matplotlib", "joblib"]
    )


def ensure_drive_mounted() -> None:
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.is_dir():
        return
    if not IN_COLAB:
        return
    from google.colab import drive

    print("Montando Google Drive...")
    drive.mount("/content/drive", force_remount=False)


def locate_dataset() -> Path:
    if IN_COLAB:
        ensure_drive_mounted()
    candidates = [
        Path(DRIVE_DATA_PATH),
        Path("/content") / DATA_FILENAME,
        Path.cwd() / DATA_FILENAME,
    ]
    for path in candidates:
        if path.is_file():
            return path.resolve()
    if IN_COLAB:
        hits = sorted(Path("/content").rglob(DATA_FILENAME))
        if hits:
            return hits[0].resolve()
    raise FileNotFoundError(
        f"No se encontró {DATA_FILENAME}. Verifique DRIVE_DATA_PATH = {DRIVE_DATA_PATH!r}"
    )


DATA_PATH = locate_dataset()
print(f"Dataset: {DATA_PATH}")

if IN_COLAB:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Salida en Drive: {DRIVE_OUTPUT_DIR}")
else:
    DRIVE_OUTPUT_DIR = Path("artefactos")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Salida local: {DRIVE_OUTPUT_DIR.resolve()}")


In [ ]:
def print_header(title: str, width: int = 72) -> None:
    line = "=" * width
    print(f"\n{line}\n{title.upper().center(width)}\n{line}")


def drop_leakage_and_ids(data: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in LEAKAGE_COLS + ID_COLS if c in data.columns]
    return data.drop(columns=cols)


def preprocess(data: pd.DataFrame, target: str = TARGET):
    processed = drop_leakage_and_ids(data).copy()
    processed[ORDINAL_COL] = (
        processed[ORDINAL_COL].astype(str).str.strip().str.title().map(BURNOUT_MAP).astype(int)
    )
    processed = pd.get_dummies(
        processed, columns=NOMINAL_COLS, drop_first=DROP_FIRST_DUMMY, dtype=int
    )
    X = processed.drop(columns=target)
    y = processed[target].astype(int)
    return X, y


def stratified_three_way_split(X, y, random_state: int = RANDOM_STATE):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def find_threshold_for_recall(y_true, y_proba, recall_target: float = RECALL_TARGET) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    precision, recall = precision[:-1], recall[:-1]
    valid = recall >= recall_target
    if not valid.any():
        return float(thresholds.min())
    f1 = 2 * precision[valid] * recall[valid] / (precision[valid] + recall[valid] + 1e-12)
    return float(thresholds[valid][np.argmax(f1)])


def compute_metrics(model_name: str, y_true, y_proba, threshold: float) -> dict:
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "Modelo": model_name,
        "Umbral": threshold,
        "Recall (C1)": recall_score(y_true, y_pred, zero_division=0),
        "Precision (C1)": precision_score(y_true, y_pred, zero_division=0),
        "F1-Score (C1)": f1_score(y_true, y_pred, zero_division=0),
        "PR-AUC": average_precision_score(y_true, y_proba),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "FN": int(fn),
        "FP": int(fp),
        "TP": int(tp),
        "TN": int(tn),
        "Cumple Recall": recall_score(y_true, y_pred, zero_division=0) >= RECALL_TARGET,
    }


df = pd.read_csv(DATA_PATH)
X, y = preprocess(df)
assert X.shape[1] == EXPECTED_N_FEATURES
X_train, X_val, X_test, y_train, y_val, y_test = stratified_three_way_split(X, y)
print_header("Datos listos")
print(f"Train {len(X_train):,} | Val {len(X_val):,} | Test {len(X_test):,} | Features {X.shape[1]}")


---
## 1. Definición y entrenamiento de los tres modelos

| Modelo | Desbalance | Escalado |
|--------|------------|----------|
| Regresión Logística | `class_weight='balanced'` | Sí (`StandardScaler`) |
| Random Forest | `class_weight='balanced_subsample'` | No |
| XGBoost | `scale_pos_weight` = ratio neg/pos en Train | No |


In [ ]:
scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

pipelines = {
    "logistic_regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1_000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "random_forest": Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            )
        ]
    ),
    "xgboost": Pipeline(
        [
            (
                "model",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=5,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    scale_pos_weight=scale_pos_weight,
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            )
        ]
    ),
}

proba_val = {}
fitted = {}
for key, pipe in pipelines.items():
    label = MODEL_LABELS[key]
    print(f"Entrenando {label}...")
    pipe.fit(X_train, y_train)
    fitted[key] = pipe
    proba_val[label] = pipe.predict_proba(X_val)[:, 1]
print_header("Entrenamiento completado")


---
## 2. Umbral óptimo en Validation (Recall ≥ 0,85)

Solo se usa **Validation** para elegir el umbral. **Test** queda reservado para la evaluación final del ganador.


In [ ]:
thresholds = {
    label: find_threshold_for_recall(y_val, proba_val[label], RECALL_TARGET)
    for label in proba_val
}

val_metrics = []
for label, proba in proba_val.items():
    row = compute_metrics(label, y_val, proba, thresholds[label])
    val_metrics.append(row)

comparison_val = pd.DataFrame(val_metrics).set_index("Modelo")
metric_cols = [
    "Umbral",
    "Recall (C1)",
    "Precision (C1)",
    "F1-Score (C1)",
    "PR-AUC",
    "ROC-AUC",
    "FN",
    "FP",
    "Cumple Recall",
]

print_header("Comparación en Validation (umbral ajustado)")
display(comparison_val[metric_cols].round(4))


---
## 3. Visualización comparativa

Gráficas pensadas para análisis rápido en defensa de proyecto: métricas lado a lado, errores FN/FP, curvas PR superpuestas y matrices de confusión.


In [ ]:
def plot_metrics_bars(df_metrics: pd.DataFrame, metrics: list, title: str) -> None:
    plot_df = df_metrics[metrics].copy()
    x = np.arange(len(plot_df))
    width = 0.22
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, col in enumerate(metrics):
        offset = (i - len(metrics) / 2 + 0.5) * width
        bars = ax.bar(x + offset, plot_df[col], width=width, label=col)
        for bar, val in zip(bars, plot_df[col]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val:.3f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=0)
    ax.set_ylim(0, 1.08)
    ax.axhline(RECALL_TARGET, color=MT_ACCENT, linestyle="--", linewidth=1.2, label=f"Recall objetivo {RECALL_TARGET}")
    ax.set_title(title, loc="left", fontweight="bold", color=MT_NAVY)
    ax.legend(frameon=False, ncol=2, fontsize=9)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_fn_fp(df_metrics: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(df_metrics))
    w = 0.35
    ax.bar(x - w / 2, df_metrics["FN"], width=w, color=MT_ACCENT, label="FN (riesgo no detectado)")
    ax.bar(x + w / 2, df_metrics["FP"], width=w, color=MT_SLATE, label="FP (falsa alarma)")
    ax.set_xticks(x)
    ax.set_xticklabels(df_metrics.index)
    ax.set_ylabel("Cantidad de casos")
    ax.set_title("Errores críticos en Validation · FN vs FP", loc="left", fontweight="bold", color=MT_NAVY)
    ax.legend(frameon=False)
    for i, row in enumerate(df_metrics.itertuples()):
        ax.text(i - w / 2, row.FN + 1, str(row.FN), ha="center", fontsize=10)
        ax.text(i + w / 2, row.FP + 1, str(row.FP), ha="center", fontsize=10)
    plt.tight_layout()
    plt.show()


def plot_pr_overlay(proba_dict: dict, y_true) -> None:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for name, proba in proba_dict.items():
        p, r, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        ax.plot(r, p, linewidth=2.2, color=MODEL_COLORS.get(name, MT_SLATE),
                label=f"{name} · PR-AUC={ap:.3f}")
    ax.axhline(np.mean(y_true), color=MT_MIST, linestyle="--", label="Clasificador aleatorio")
    ax.axvline(RECALL_TARGET, color=MT_ACCENT, linestyle=":", label=f"Recall objetivo {RECALL_TARGET}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Curvas Precision-Recall superpuestas (Validation)", loc="left", fontweight="bold", color=MT_NAVY)
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


def plot_confusion_grid(proba_dict: dict, y_true, thresholds: dict) -> None:
    n = len(proba_dict)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.3))
    if n == 1:
        axes = [axes]
    cmap = LinearSegmentedColormap.from_list("cm", ["#F8FAFC", MT_NAVY])
    for ax, (name, proba) in zip(axes, proba_dict.items()):
        thr = thresholds[name]
        y_pred = (proba >= thr).astype(int)
        ConfusionMatrixDisplay(
            confusion_matrix(y_true, y_pred, labels=[0, 1]),
            display_labels=["Sin riesgo", "En riesgo"],
        ).plot(ax=ax, cmap=cmap, colorbar=False, values_format=",d")
        ax.set_title(f"{name}\nUmbral {thr:.3f}", loc="left", fontweight="bold")
        ax.grid(False)
    fig.suptitle("Matrices de confusión · Validation", x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
    plt.tight_layout()
    plt.show()


plot_metrics_bars(
    comparison_val,
    ["Recall (C1)", "F1-Score (C1)", "Precision (C1)", "PR-AUC", "ROC-AUC"],
    "Métricas principales · Validation (umbral ajustado)",
)
plot_fn_fp(comparison_val)
plot_pr_overlay(proba_val, y_val)
plot_confusion_grid(proba_val, y_val, thresholds)

# Heatmap de métricas normalizadas (0-1) para lectura rápida
heat_cols = ["Recall (C1)", "Precision (C1)", "F1-Score (C1)", "PR-AUC", "ROC-AUC"]
heat = comparison_val[heat_cols]
fig, ax = plt.subplots(figsize=(7, 3.8))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set_title("Mapa de calor · métricas por modelo", loc="left", fontweight="bold", color=MT_NAVY)
plt.tight_layout()
plt.show()


---
## 4. Selección del modelo ganador

**Criterio (Validation):**
1. Debe cumplir `Recall (C1) ≥ 0,85`.
2. Entre los que cumplen: mayor **F1-Score (C1)**.
3. Empate: mayor **PR-AUC**.


In [ ]:
def select_winner(df_metrics: pd.DataFrame) -> str:
    eligible = df_metrics[df_metrics["Cumple Recall"]].copy()
    if eligible.empty:
        print("Ningún modelo cumple Recall en Validation; se elige el de mayor Recall.")
        eligible = df_metrics.copy()
    eligible = eligible.sort_values(["F1-Score (C1)", "PR-AUC"], ascending=False)
    return eligible.index[0]


winner = select_winner(comparison_val)
winner_key = next(k for k, v in MODEL_LABELS.items() if v == winner)
winner_threshold = thresholds[winner]
winner_model = fitted[winner_key]

print_header("Modelo seleccionado")
print(f"Ganador: {winner}")
print(f"Umbral en Validation: {winner_threshold:.4f}")


---
## 5. Evaluación final en Test (una sola vez)

Se aplica el umbral aprendido en Validation al conjunto de **Test**.


In [ ]:
proba_test = winner_model.predict_proba(X_test)[:, 1]
test_metrics = compute_metrics(winner, y_test, proba_test, winner_threshold)
test_df = pd.DataFrame([test_metrics]).set_index("Modelo")

print_header(f"Evaluación final en Test · {winner}")
display(test_df[metric_cols].round(4))
print(classification_report(y_test, (proba_test >= winner_threshold).astype(int),
                          target_names=["Sin riesgo", "En riesgo"], digits=4))

cm_cmap = LinearSegmentedColormap.from_list("cm", ["#F8FAFC", MT_NAVY])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, (proba_test >= winner_threshold).astype(int), labels=[0, 1]),
    display_labels=["Sin riesgo", "En riesgo"],
).plot(ax=axes[0], cmap=cm_cmap, colorbar=False, values_format=",d")
axes[0].set_title("Matriz de confusión · Test", loc="left", fontweight="bold")
precision, recall, _ = precision_recall_curve(y_test, proba_test)
axes[1].plot(recall, precision, color=MT_NAVY, linewidth=2,
             label=f"PR-AUC = {test_metrics['PR-AUC']:.3f}")
axes[1].axvline(RECALL_TARGET, color=MT_ACCENT, linestyle=":", label=f"Recall objetivo {RECALL_TARGET}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Curva PR · Test", loc="left", fontweight="bold")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()


---
## 6. Importancia de variables (interpretación rápida)

Comparación de señales más relevantes según el tipo de modelo.


In [ ]:
feature_names = X_train.columns.tolist()

# Regresión Logística: coeficientes estandarizados
lr_coef = pd.Series(
    fitted["logistic_regression"].named_steps["model"].coef_[0], index=feature_names
)
lr_coef = lr_coef.sort_values(key=np.abs, ascending=False).head(10)

# Random Forest
rf_imp = pd.Series(
    fitted["random_forest"].named_steps["model"].feature_importances_, index=feature_names
).sort_values(ascending=False).head(10)

# XGBoost
xgb_imp = pd.Series(
    fitted["xgboost"].named_steps["model"].feature_importances_, index=feature_names
).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, series, title in zip(
    axes,
    [lr_coef, rf_imp, xgb_imp],
    ["Top 10 · Regresión Logística (|coef|)", "Top 10 · Random Forest", "Top 10 · XGBoost"],
):
    plot_series = series.sort_values(key=np.abs) if "Regresión" in title else series.sort_values()
    plot_series.plot(kind="barh", ax=ax, color=MT_NAVY, edgecolor=MT_INK, linewidth=0.4)
    ax.set_title(title, loc="left", fontweight="bold", color=MT_NAVY)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()


---
## 7. Guardar en Google Drive

Se exportan tabla comparativa, métricas de test, modelo ganador y gráficas PNG en `Artefactos/`.


In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")

comparison_val.round(4).to_csv(DRIVE_OUTPUT_DIR / f"comparacion_validation_{stamp}.csv")
test_df.round(4).to_csv(DRIVE_OUTPUT_DIR / f"evaluacion_test_{winner}_{stamp}.csv")

bundle = {
    "winner_key": winner_key,
    "winner_label": winner,
    "threshold": winner_threshold,
    "feature_names": feature_names,
    "metrics_validation": comparison_val.reset_index().to_dict(orient="records"),
    "metrics_test": test_metrics,
    "random_state": RANDOM_STATE,
    "recall_target": RECALL_TARGET,
}
joblib.dump(bundle, DRIVE_OUTPUT_DIR / f"model_bundle_{winner_key}_{stamp}.joblib")
joblib.dump(winner_model, DRIVE_OUTPUT_DIR / f"pipeline_{winner_key}_{stamp}.joblib")

# Re-exportar gráficas clave como PNG
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(11, 5))
metrics_plot = ["Recall (C1)", "F1-Score (C1)", "PR-AUC"]
for i, m in enumerate(metrics_plot):
    ax.plot(comparison_val.index, comparison_val[m], marker="o", linewidth=2, label=m)
ax.axhline(RECALL_TARGET, color=MT_ACCENT, linestyle="--", label="Recall objetivo")
ax.set_title("Resumen comparativo exportado")
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(DRIVE_OUTPUT_DIR / f"resumen_comparativo_{stamp}.png", dpi=150)
plt.show()

print_header("Archivos guardados")
for path in sorted(DRIVE_OUTPUT_DIR.glob(f"*{stamp}*")):
    print(path)


---
## 8. Conclusiones para el informe

Use la tabla de Validation para argumentar por qué el ganador equilibra **detección de riesgo (Recall)** y **precisión operativa (F1 / PR-AUC)**. Cite el resultado de Test como evidencia de generalización.

**Próximo paso:** integrar `pipeline_*.joblib` y el umbral en el prototipo Streamlit (Entregable final).
